# Remaining priority-1 selection runs on Kaggle

Runs the 10 RGB-D runs that Colab did not finish, with the same code (cloned from GitHub) and the same settings.

**Before the first run**
1. Make a private Kaggle dataset with two files from `MyDrive/rgbx/`: `nyuv2_prepared.zip` and the `splits_sel` folder (zip it first). Either zipped or unzipped is fine.
2. In this notebook: **Add Input** → your dataset. **Settings** → Accelerator **GPU T4 x2**, Internet **on** (needs a phone-verified account).

**Each part**: set `PART` in cell 3, then **Save Version → Save & Run All (Commit)**. It runs on Kaggle's servers (laptop can be off), about 3 hours per part.
When it finishes, open the version's **Output** tab and download `runs/results.csv`.

In [ ]:
# Cell 1: check the GPU, get the code
import os, glob, subprocess, torch
def sh(cmd):
    print('$', cmd, flush=True); return subprocess.run(cmd, shell=True).returncode

print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'Turn on the GPU: Settings > Accelerator > GPU T4 x2'
torch.ones(1).cuda().sum().item()   # an error here means this GPU is not supported: choose GPU T4 x2

CODE = '/tmp/code'
sh(f'rm -rf {CODE} && git clone -q https://github.com/talha-ansarii/btech-project.git {CODE}')
sh(f'cd {CODE} && git log --oneline -1')
sh(f'pip -q install -r {CODE}/requirements.txt')

In [ ]:
# Cell 2: find the data and the split files in your Kaggle dataset
INPUT = '/kaggle/input'
def find(name):
    hits = glob.glob(f'{INPUT}/**/{name}', recursive=True) + glob.glob(f'/tmp/unz/**/{name}', recursive=True)
    if not hits:   # Kaggle sometimes keeps uploaded zips as zips: unpack them once and look again
        for z in glob.glob(f'{INPUT}/**/*.zip', recursive=True):
            sh(f'mkdir -p /tmp/unz && unzip -q -o "{z}" -d /tmp/unz')
        hits = glob.glob(f'/tmp/unz/**/{name}', recursive=True)
    assert hits, f'{name} not found: add your rgbx dataset with "Add Input"'
    return os.path.dirname(hits[0])

DATA = find('meta.json')
SPLITS = find('r2-entropy_frac0.10_seed0.json')
RUNS = '/kaggle/working/runs'
print('DATA', DATA, '| SPLITS', SPLITS)
print(len(os.listdir(f'{DATA}/train/rgb')), len(os.listdir(f'{DATA}/test/rgb')), len(os.listdir(SPLITS)), '(expect 795 654 30)')

In [ ]:
# Cell 3: the remaining priority-1 runs. Set PART, then Save Version > Save & Run All (Commit)
PART = 1   # 1 the first time, then change to 2 and save a new version
JOBS = {1: [('r2-cdiv', 2), ('r2-entropy', 0), ('r2-entropy', 1), ('r2-entropy', 2), ('r2-coreset', 0)],
        2: [('r2-coreset', 1), ('r2-coreset', 2), ('r2-mcomp', 0), ('r2-mcomp', 1), ('r2-mcomp', 2)]}
for method, seed in JOBS[PART]:
    sh(f'python {CODE}/run_selection.py --data {DATA} --out {RUNS} --splits-dir {SPLITS} '
       f'--jobs {method}:0.10 --seeds {seed}')
sh(f'cat {RUNS}/results.csv')